# Where did 128 bits go?

The Coldcard entropy collapse, demonstrated by **counting** — not by asserting.

This notebook does **not** reimplement Yasmarang and does **not** attack any device.
It models the *shape* of the failure: a generator whose OUTPUT is 128 bits wide,
but whose STATE is small — so the number of distinct outputs it can ever produce
equals the size of that state, not 2^128.

Companion to the video. Repo: `github.com/nikshev/unstuck`


## 1. The output is always 128 bits wide — that is what you see


In [ ]:
import hashlib, struct

def prng_output(state: int) -> bytes:
    """A 128-bit-WIDE output derived deterministically from a small state.
    Any deterministic PRNG has this property; the algorithm does not matter."""
    return hashlib.sha256(struct.pack('>Q', state)).digest()[:16]

for s in (3, 10, 16):
    out = prng_output(s)
    print(f'state={s:<6} -> {out.hex()}   ({len(out)*8} bits wide)')

print()
print('Every one of these looks like a full-strength seed. None of them is.')


## 2. Now count how many different outputs are actually reachable

This is the whole argument. Run it and watch the last column.


In [ ]:
def count_distinct(state_bits: int) -> int:
    return len({prng_output(s) for s in range(1 << state_bits)})

print(f"{'state bits':>11} {'possible states':>18} {'distinct outputs':>18} {'real entropy':>14}")
for bits in (4, 8, 12, 16, 20):
    n = count_distinct(bits)
    print(f'{bits:>11} {1 << bits:>18,} {n:>18,} {str(bits)+" bits":>14}')

print()
print('Distinct outputs == the state space. The 128-bit width is decoration.')


## 3. So where did the Coldcard bits go?

Yasmarang was seeded **once** from three things:

* the low 32 bits of the **chip UID** — fixed in silicon, *not secret*
* the **SysTick** counter — time since power-on
* two **RTC** registers — wall-clock time

The UID is not secret, but an attacker still has to *enumerate* it, so it costs
work: 32 bits. The timers are not secret either — their cost is only however much
uncertainty the attacker cannot bound.


In [ ]:
print(f"{'attacker can pin the setup window to...':<42}{'timer bits left':>16}{'TOTAL':>8}")
for label, timer_bits in (('the exact minute', 4),
                          ('roughly the hour', 8),
                          ('roughly the day', 16),
                          ('nothing at all (full 64-bit timers)', 64)):
    print(f'{label:<42}{timer_bits:>16}{32 + timer_bits:>8}')

print()
print('Coinkite / Block Engineering put the real figure at ~40 bits for Mk2/Mk3.')
print('That is THEIR measurement, not a number this notebook derives — but notice')
print("it lands exactly where '32 bits of UID + ~8 bits of residual timer' does.")
print()
print('Designed:  128 bits')
print('Actual:    ~40 bits')
print('Lost:       88 bits  =  88 HALVINGS of the attacker work')


## 4. What one bit is worth

The intuition everyone gets wrong: 40 is not "a third as safe" as 128.


In [ ]:
print(f"{'bits':>6} {'search space':>44}")
for b in (40, 48, 64, 80, 96, 112, 128):
    print(f'{b:>6} {1 << b:>44,}')

print()
print('2^128 / 2^40 = 2^88 times easier.')
print(f'2^88 = {1 << 88:,}')


## 5. On a GPU — what 2^40 actually feels like

Runtime → Change runtime type → **T4 GPU**, then run the two cells below.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


In [ ]:
import numpy as np, time
from numba import cuda

@cuda.jit
def walk(out, base, n):
    i = cuda.grid(1)
    if i < n:
        # a deterministic mix per candidate - stands in for 'derive and compare'
        x = np.uint64(base + i)
        x ^= x >> np.uint64(33); x *= np.uint64(0xff51afd7ed558ccd)
        x ^= x >> np.uint64(33)
        if x == np.uint64(0):
            out[0] += 1

N = 1 << 26
out = cuda.to_device(np.zeros(1, dtype=np.uint64))
threads = 256; blocks = (N + threads - 1) // threads

walk[blocks, threads](out, np.uint64(0), N)   # warm-up / JIT
cuda.synchronize()

t0 = time.time()
walk[blocks, threads](out, np.uint64(0), N)
cuda.synchronize()
el = time.time() - t0

rate = N / el
print(f'walked {N:,} candidates in {el*1000:.1f} ms')
print(f'rate: {rate:,.0f} candidates/second')
print()
space40, space128 = 1 << 40, 1 << 128
print(f'2^40  at this rate: {space40/rate:,.1f} seconds')
print(f'2^128 at this rate: {space128/rate/31557600:.3g} years')
print()
print('NOTE: this counts a CHEAP operation, like the vanity-address demo.')
print('A real BIP39 seed check is ~2,000x dearer - see the 40 Bits episode.')
print('The point here is the RATIO between 2^40 and 2^128, not either time.')
